# Sprint 8 — Megaline: recomendação de planos

Projeto de aprendizado de máquina desenvolvido no programa de Ciência de Dados da TripleTen.

A Megaline quer analisar o comportamento mensal dos clientes e recomendar um dos planos atuais: **Smart** ou **Ultra**. O objetivo é construir um classificador com a maior acurácia possível e alcançar pelo menos **0,75 no conjunto de teste**.

Cada observação contém:

| Coluna | Descrição |
|---|---|
| `calls` | Número de chamadas |
| `minutes` | Duração total das chamadas em minutos |
| `messages` | Número de mensagens de texto |
| `mb_used` | Tráfego de internet utilizado em MB |
| `is_ultra` | Plano atual: Ultra = 1; Smart = 0 |


## 1. Abrir e examinar os dados


In [1]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('data/users_behavior.csv'),
    Path('/datasets/users_behavior.csv'),
    Path('users_behavior.csv')
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError('O arquivo users_behavior.csv não foi encontrado.')

dados = pd.read_csv(data_path)
print(f'Arquivo carregado: {data_path}')


Arquivo carregado: data/users_behavior.csv


In [2]:
dados.head()


,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [3]:
dados.shape


(3214, 5)

In [4]:
dados.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [5]:
print('Valores ausentes por coluna:')
print(dados.isna().sum())
print(f'\nLinhas inteiramente duplicadas: {dados.duplicated().sum()}')


Valores ausentes por coluna:
calls       0
minutes     0
messages    0
mb_used     0
is_ultra    0
dtype: int64

Linhas inteiramente duplicadas: 0


In [6]:
plan_summary = pd.DataFrame({
    'registros': dados['is_ultra'].value_counts().sort_index(),
    'proporcao': dados['is_ultra'].value_counts(normalize=True).sort_index()
})

plan_summary.index = ['Smart (0)', 'Ultra (1)']
plan_summary


,registros,proporcao
Smart (0),2229,0.693528
Ultra (1),985,0.306472


### Conclusão da análise inicial

O conjunto possui 3.214 registros e 5 colunas. Não foram encontrados valores ausentes nem linhas inteiramente duplicadas. As quatro colunas de comportamento são numéricas e `is_ultra` é a variável que será prevista. Aproximadamente 69,35% dos registros pertencem ao plano Smart e 30,65% ao plano Ultra.


## 2. Divisão dos dados


In [7]:
from sklearn.model_selection import train_test_split

features = dados.drop('is_ultra', axis=1)
target = dados['is_ultra']

features_train, features_temp, target_train, target_temp = train_test_split(
    features,
    target,
    test_size=0.40,
    random_state=12345,
    stratify=target
)

features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp,
    target_temp,
    test_size=0.50,
    random_state=12345,
    stratify=target_temp
)


In [8]:
print('Treinamento:', features_train.shape, target_train.shape)
print('Validação:', features_valid.shape, target_valid.shape)
print('Teste:', features_test.shape, target_test.shape)


Treinamento: (1928, 4) (1928,)
Validação: (643, 4) (643,)
Teste: (643, 4) (643,)


In [9]:
class_proportions = pd.DataFrame({
    'Dados completos': target.value_counts(normalize=True).sort_index(),
    'Treinamento': target_train.value_counts(normalize=True).sort_index(),
    'Validação': target_valid.value_counts(normalize=True).sort_index(),
    'Teste': target_test.value_counts(normalize=True).sort_index()
}).T

class_proportions.columns = ['Smart (0)', 'Ultra (1)']
(class_proportions * 100).round(2)


,Smart (0),Ultra (1)
Dados completos,69.35,30.65
Treinamento,69.35,30.65
Validação,69.36,30.64
Teste,69.36,30.64


### Conclusão da divisão dos dados

Os dados foram divididos de forma estratificada em 60% para treinamento, 20% para validação e 20% para teste. Essa divisão prioriza o treinamento do modelo, com 1.928 registros, e mantém 643 registros em cada conjunto de avaliação, quantidade suficiente para obter estimativas de acurácia razoavelmente estáveis.

A comparação das proporções confirmou que os três conjuntos preservaram aproximadamente a distribuição original dos planos, com cerca de 69,4% dos registros pertencentes ao plano Smart e 30,6% ao plano Ultra. O conjunto de teste será utilizado somente na avaliação final do modelo escolhido.


## 3. Investigação dos modelos

### 3.1 Árvore de decisão

O primeiro modelo avaliado será uma árvore de decisão. O hiperparâmetro `max_depth` será alterado para identificar a profundidade que produz a maior acurácia no conjunto de validação.


In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

best_tree_model = None
best_tree_accuracy = 0
best_tree_depth = 0

for depth in range(1, 11):
    tree_model = DecisionTreeClassifier(
        random_state=12345,
        max_depth=depth
    )

    tree_model.fit(features_train, target_train)

    train_predictions = tree_model.predict(features_train)
    valid_predictions = tree_model.predict(features_valid)

    train_accuracy = accuracy_score(target_train, train_predictions)
    valid_accuracy = accuracy_score(target_valid, valid_predictions)

    print(
        f'Profundidade: {depth} | '
        f'Treinamento: {train_accuracy:.4f} | '
        f'Validação: {valid_accuracy:.4f}'
    )

    if valid_accuracy > best_tree_accuracy:
        best_tree_model = tree_model
        best_tree_accuracy = valid_accuracy
        best_tree_depth = depth


Profundidade: 1 | Treinamento: 0.7547 | Validação: 0.7403
Profundidade: 2 | Treinamento: 0.7759 | Validação: 0.7729
Profundidade: 3 | Treinamento: 0.7910 | Validação: 0.7776
Profundidade: 4 | Treinamento: 0.7972 | Validação: 0.7543
Profundidade: 5 | Treinamento: 0.8138 | Validação: 0.7854
Profundidade: 6 | Treinamento: 0.8283 | Validação: 0.7745
Profundidade: 7 | Treinamento: 0.8413 | Validação: 0.7869
Profundidade: 8 | Treinamento: 0.8558 | Validação: 0.8025
Profundidade: 9 | Treinamento: 0.8719 | Validação: 0.7823
Profundidade: 10 | Treinamento: 0.8874 | Validação: 0.7729


In [11]:
print('Melhor profundidade:', best_tree_depth)
print(f'Melhor acurácia de validação: {best_tree_accuracy:.4f}')


Melhor profundidade: 8
Melhor acurácia de validação: 0.8025


#### Resultado da árvore de decisão

A árvore de decisão apresentou sua melhor acurácia de validação com `max_depth=8`, atingindo 0,8025. Nessa configuração, a acurácia de treinamento foi 0,8558.

Nas profundidades 9 e 10, a acurácia de treinamento continuou aumentando, mas a acurácia de validação diminuiu. Esse comportamento indica sobreajuste, pois o modelo passou a se adaptar excessivamente aos dados de treinamento. Portanto, entre as configurações avaliadas, a profundidade 8 apresentou o melhor desempenho.


### 3.2 Floresta aleatória

O segundo modelo avaliado será uma floresta aleatória. Serão testados diferentes números de árvores e diferentes profundidades máximas. A melhor configuração será escolhida com base na acurácia do conjunto de validação.


In [12]:
from sklearn.ensemble import RandomForestClassifier

best_forest_model = None
best_forest_accuracy = 0
best_forest_estimators = 0
best_forest_depth = 0
forest_results = []

for estimators in [10, 25, 50, 75, 100]:
    for depth in range(2, 11):
        forest_model = RandomForestClassifier(
            random_state=12345,
            n_estimators=estimators,
            max_depth=depth
        )

        forest_model.fit(features_train, target_train)

        train_predictions = forest_model.predict(features_train)
        valid_predictions = forest_model.predict(features_valid)

        train_accuracy = accuracy_score(target_train, train_predictions)
        valid_accuracy = accuracy_score(target_valid, valid_predictions)

        forest_results.append({
            'n_estimators': estimators,
            'max_depth': depth,
            'train_accuracy': train_accuracy,
            'valid_accuracy': valid_accuracy
        })

        if valid_accuracy > best_forest_accuracy:
            best_forest_model = forest_model
            best_forest_accuracy = valid_accuracy
            best_forest_estimators = estimators
            best_forest_depth = depth

forest_results_df = pd.DataFrame(forest_results)


In [13]:
print('Combinações avaliadas:', len(forest_results_df))
print(
    'Combinações repetidas:',
    forest_results_df.duplicated(
        subset=['n_estimators', 'max_depth']
    ).sum()
)

forest_results_df.sort_values(
    by='valid_accuracy',
    ascending=False
).head(10)


Combinações avaliadas: 45
Combinações repetidas: 0


,n_estimators,max_depth,train_accuracy,valid_accuracy
25,50,9,0.883299,0.818040
32,75,7,0.858921,0.816485
34,75,9,0.884336,0.816485
35,75,10,0.891079,0.816485
44,100,10,0.892116,0.814930
42,100,8,0.875000,0.814930
43,100,9,0.883817,0.813375
26,50,10,0.891598,0.813375
41,100,7,0.859440,0.813375
15,25,8,0.870332,0.811820


In [14]:
print('Melhor número de árvores:', best_forest_estimators)
print('Melhor profundidade:', best_forest_depth)
print(f'Melhor acurácia de validação: {best_forest_accuracy:.4f}')


Melhor número de árvores: 50
Melhor profundidade: 9
Melhor acurácia de validação: 0.8180


#### Resultado da floresta aleatória

Foram avaliadas 45 combinações de hiperparâmetros. A melhor floresta aleatória utilizou 50 árvores, com profundidade máxima igual a 9, e alcançou acurácia de 0,8180 no conjunto de validação. Esse resultado superou em 0,0155, ou 1,55 ponto percentual, a acurácia de 0,8025 obtida pela árvore de decisão.

Os melhores modelos alcançaram aproximadamente 0,88 a 0,89 de acurácia no treinamento e cerca de 0,82 na validação. Essa diferença indica um nível moderado de sobreajuste: o aumento da profundidade melhora o ajuste aos dados de treinamento, mas não produz o mesmo ganho na validação.

As dez melhores configurações ficaram separadas por aproximadamente 0,005 de acurácia. Como o conjunto de validação possui 643 registros, uma única previsão correta representa aproximadamente 0,0016 de acurácia. Portanto, as diferenças entre várias das melhores configurações correspondem a poucos usuários e não indicam uma superioridade muito ampla entre elas.

Algumas configurações com profundidade 10, o limite máximo investigado, também apareceram entre os melhores resultados. Um estudo futuro poderia explorar profundidades maiores. Dentro do intervalo avaliado neste projeto, a configuração com 50 árvores e profundidade 9 apresentou o melhor resultado.


### 3.3 Regressão logística

O terceiro modelo avaliado será a regressão logística. Antes do treinamento, as características serão padronizadas, pois possuem escalas muito diferentes. Serão testados diferentes valores do hiperparâmetro de regularização `C`.


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_train_scaled = scaler.fit_transform(features_train)
features_valid_scaled = scaler.transform(features_valid)


In [16]:
best_logistic_model = None
best_logistic_accuracy = 0
best_logistic_c = 0

for c_value in [0.001, 0.01, 0.1, 1, 10, 100]:
    logistic_model = LogisticRegression(
        random_state=12345,
        solver='liblinear',
        C=c_value,
        max_iter=1000
    )

    logistic_model.fit(features_train_scaled, target_train)

    train_predictions = logistic_model.predict(features_train_scaled)
    valid_predictions = logistic_model.predict(features_valid_scaled)

    train_accuracy = accuracy_score(target_train, train_predictions)
    valid_accuracy = accuracy_score(target_valid, valid_predictions)

    print(
        f'C: {c_value} | '
        f'Treinamento: {train_accuracy:.4f} | '
        f'Validação: {valid_accuracy:.4f}'
    )

    if valid_accuracy > best_logistic_accuracy:
        best_logistic_model = logistic_model
        best_logistic_accuracy = valid_accuracy
        best_logistic_c = c_value


C: 0.001 | Treinamento: 0.7427 | Validação: 0.7341
C: 0.01 | Treinamento: 0.7510 | Validação: 0.7434
C: 0.1 | Treinamento: 0.7495 | Validação: 0.7418
C: 1 | Treinamento: 0.7505 | Validação: 0.7387
C: 10 | Treinamento: 0.7510 | Validação: 0.7387
C: 100 | Treinamento: 0.7510 | Validação: 0.7387


In [17]:
print('Melhor valor de C:', best_logistic_c)
print(f'Melhor acurácia de validação: {best_logistic_accuracy:.4f}')


Melhor valor de C: 0.01
Melhor acurácia de validação: 0.7434


#### Resultado da regressão logística

A melhor regressão logística utilizou `C=0.01` e alcançou acurácia de 0,7434 no conjunto de validação. O resultado ficou abaixo do limite de 0,75 e também foi inferior aos resultados da árvore de decisão e da floresta aleatória.

As acurácias de treinamento e validação permaneceram próximas, indicando que o modelo não apresentou forte sobreajuste. Entretanto, a regressão logística não conseguiu representar os padrões dos dados tão bem quanto os modelos baseados em árvores.


### 3.4 Comparação dos modelos


In [18]:
model_comparison = pd.DataFrame({
    'modelo': [
        'Árvore de decisão',
        'Floresta aleatória',
        'Regressão logística'
    ],
    'melhor configuração': [
        'max_depth=8',
        'n_estimators=50, max_depth=9',
        'C=0.01'
    ],
    'acurácia de validação': [
        best_tree_accuracy,
        best_forest_accuracy,
        best_logistic_accuracy
    ]
})

model_comparison.sort_values(
    by='acurácia de validação',
    ascending=False
).reset_index(drop=True)


,modelo,melhor configuração,acurácia de validação
0,Floresta aleatória,"n_estimators=50, max_depth=9",0.818040
1,Árvore de decisão,max_depth=8,0.802488
2,Regressão logística,C=0.01,0.743390


A floresta aleatória apresentou a maior acurácia no conjunto de validação. Portanto, ela foi selecionada para a avaliação final no conjunto de teste.


## 4. Avaliação do modelo no conjunto de teste

### Treinamento do modelo final

A acurácia de validação de 0,8180 foi obtida pelo modelo treinado somente com os 60% destinados ao treinamento. Após a escolha dos hiperparâmetros, os conjuntos de treinamento e validação foram reunidos para que o modelo final pudesse aprender com 80% dos dados disponíveis.

O modelo final manteve a configuração escolhida, com 50 árvores e profundidade máxima igual a 9, mas foi treinado novamente com o conjunto ampliado. Em seguida, foi avaliado uma única vez no conjunto de teste.


In [19]:
features_final_train = pd.concat(
    [features_train, features_valid],
    ignore_index=True
)

target_final_train = pd.concat(
    [target_train, target_valid],
    ignore_index=True
)

print(features_final_train.shape)
print(target_final_train.shape)


(2571, 4)
(2571,)


In [20]:
final_model = RandomForestClassifier(
    random_state=12345,
    n_estimators=best_forest_estimators,
    max_depth=best_forest_depth
)

final_model.fit(features_final_train, target_final_train)
test_predictions = final_model.predict(features_test)
test_accuracy = accuracy_score(target_test, test_predictions)

test_correct = int((test_predictions == target_test).sum())
test_incorrect = len(target_test) - test_correct

print(f'Acurácia no conjunto de teste: {test_accuracy:.4f}')
print(f'Previsões corretas: {test_correct}')
print(f'Previsões incorretas: {test_incorrect}')


Acurácia no conjunto de teste: 0.8305
Previsões corretas: 534
Previsões incorretas: 109


## 5. Verificação de sanidade do modelo

Para verificar se o modelo final realmente aprendeu padrões dos dados, seu desempenho será comparado ao de um classificador básico que sempre prevê a classe mais frequente.


In [21]:
from sklearn.dummy import DummyClassifier

dummy_model = DummyClassifier(strategy='most_frequent')
dummy_model.fit(features_final_train, target_final_train)
dummy_predictions = dummy_model.predict(features_test)

dummy_accuracy = accuracy_score(target_test, dummy_predictions)
accuracy_improvement = test_accuracy - dummy_accuracy

print(f'Acurácia do modelo final: {test_accuracy:.4f}')
print(f'Acurácia do modelo básico: {dummy_accuracy:.4f}')
print(f'Melhoria sobre o modelo básico: {accuracy_improvement:.4f}')


Acurácia do modelo final: 0.8305
Acurácia do modelo básico: 0.6936
Melhoria sobre o modelo básico: 0.1369


### Resultado da verificação de sanidade

O modelo básico, que sempre prevê o plano Smart por ser a classe mais frequente, obteve acurácia de 0,6936. Já a floresta aleatória alcançou 0,8305 no mesmo conjunto de teste.

A diferença foi de 0,1369, equivalente a 13,69 pontos percentuais. Isso demonstra que a floresta aleatória aprendeu relações úteis entre o comportamento dos usuários e seus planos, apresentando desempenho claramente superior ao de uma estratégia baseada apenas na classe mais frequente.


## 6. Conclusão geral

O projeto utilizou dados comportamentais de 3.214 usuários para desenvolver um modelo capaz de recomendar um dos planos mais recentes da Megaline: Smart ou Ultra. O conjunto não apresentou valores ausentes nem linhas inteiramente duplicadas e já estava preparado para a etapa de modelagem.

Os dados foram divididos de forma estratificada em 60% para treinamento, 20% para validação e 20% para teste. Foram avaliados três modelos de classificação. A árvore de decisão atingiu 0,8025 na validação, a floresta aleatória alcançou 0,8180 e a regressão logística obteve 0,7434.

A floresta aleatória apresentou o melhor resultado na validação e foi selecionada como modelo final. Após ser treinada novamente com os conjuntos de treinamento e validação reunidos, atingiu acurácia de 0,8305 no conjunto de teste, superando o limite mínimo de 0,75 estabelecido para o projeto.

Na verificação de sanidade, o classificador básico obteve acurácia de 0,6936. A floresta aleatória apresentou uma melhoria de 13,69 pontos percentuais, confirmando que o modelo aprendeu padrões relevantes nos dados e não se limitou a prever o plano mais frequente.

Portanto, o objetivo do projeto foi alcançado. O modelo pode auxiliar a Megaline na classificação dos planos Smart ou Ultra com base no comportamento mensal dos clientes, respeitadas as limitações dos dados e a necessidade de validação antes de uma aplicação real.
